# Milestone 7: Wide-Column Database Architecture & Showcase

**Project:** Chain Reaction — High-Dimensional Distributed AML Detection Engine  
**Domain:** Financial Crime & Graph Analytics  
**Technologies:** Apache Cassandra (CQL), Apache Spark SQL, Databricks Delta Lake / Parquet, PySpark

---

## 1. Executive Database Summary

Modern Anti-Money Laundering (AML) monitoring systems must process millions of continuous transactions per day while performing complex graph traversals and maintaining real-time compliance profile updates.

Traditional Relational Databases (RDBMS) fail at this scale due to:
1. **Lock Contention:** B-Tree indexing and row-level write locks throttle high-frequency financial writes.
2. **Expensive Joins:** Graph traversal (e.g. ego-subgraph reconstruction) requires recursive, multi-table joins.
3. **Rigid Schemas:** Adding newly discovered ML features requires costly table-locking `ALTER TABLE` migrations.

This project implements a **Wide-Column NoSQL Storage Model** (modeled after **Apache Cassandra / ScyllaDB** and implemented locally on **Databricks Delta Lake / Parquet**):
- **Partition Keys:** Distribute account histories uniformly across storage nodes via Murmur3 hash partitioning ($O(1)$ routing).
- **Clustering Keys:** Store transactions within each partition in strictly reverse chronological order (`timestamp DESC`), enabling instant temporal slice scans without full table scans.
- **Wide Row Feature Store:** Consolidates 18+ high-dimensional features per account entity without NULL storage penalties.
- **Zero-Downtime Schema Evolution:** Ingests dynamic ML risk scores and watchlist updates instantaneously.


In [1]:
import os
import sys
import asyncio
import time

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))
os.environ['PYTHONPATH'] = os.path.abspath('..') + os.pathsep + os.environ.get('PYTHONPATH', '')

from src.spark_features import _init_windows_hadoop
_init_windows_hadoop()

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName('ChainReaction-DB-Showcase')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark Database Engine Initialized.')
print(f'Engine Version: {spark.version} | Driver Threads: {spark.sparkContext.defaultParallelism}')


C:\Projects\MULE-T~2\CHAIN-~1\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Database Engine Initialized.
Engine Version: 4.2.0 | Driver Threads: 12


## 2. Cassandra CQL Schema Architecture (`docs/cassandra_schema.cql`)

Below is the production DDL specification for the two primary tables in the `chain_reaction_aml` keyspace:

```sql
CREATE KEYSPACE IF NOT EXISTS chain_reaction_aml
WITH replication = {
    'class': 'NetworkTopologyStrategy',
    'us-east-1': 3,
    'eu-west-1': 3
};

-- 1. TIME-SERIES TRANSACTION EVENTS
-- Partition Key: account_id (Uniform Murmur3 hash distribution)
-- Clustering Key: event_timestamp DESC, transaction_id ASC (Strict local order)
CREATE TABLE IF NOT EXISTS chain_reaction_aml.transaction_events (
    account_id              TEXT,
    event_timestamp         TIMESTAMP,
    transaction_id          UUID,
    direction               TEXT,
    counterparty_account    TEXT,
    amount_paid_usd         DOUBLE,
    amount_received_usd     DOUBLE,
    payment_currency        TEXT,
    payment_format          TEXT,
    is_laundering           INT,
    PRIMARY KEY ((account_id), event_timestamp, transaction_id)
) WITH CLUSTERING ORDER BY (event_timestamp DESC, transaction_id ASC)
  AND default_time_to_live = 31536000;

-- 2. HIGH-DIMENSIONAL ACCOUNT PROFILE (WIDE ROW)
-- Partition Key: account_id
CREATE TABLE IF NOT EXISTS chain_reaction_aml.account_profiles (
    account_id                  TEXT PRIMARY KEY,
    in_degree                   INT,
    out_degree                  INT,
    total_in_usd                DOUBLE,
    total_out_usd               DOUBLE,
    unique_counterparties       INT,
    is_sanctioned               BOOLEAN,
    sanctions_risk_score        DOUBLE,
    jurisdiction_code           TEXT,
    jurisdiction_risk_score     DOUBLE,
    max_txns_per_hour           INT,
    avg_txns_per_hour           DOUBLE,
    is_laundering               INT,
    updated_at                  TIMESTAMP
);
```


In [2]:
# Inspect the physical storage directory of our Wide-Column Tables
storage_root = os.path.abspath('../outputs/wide_column_store')
tx_path = os.path.join(storage_root, 'transaction_events')
prof_path = os.path.join(storage_root, 'account_profiles')

print(f'Wide Column Storage Root: {storage_root}')

# Read persisted Parquet tables into Spark
tx_df = spark.read.parquet(tx_path)
prof_df = spark.read.parquet(prof_path)

tx_df.createOrReplaceTempView('aml_transaction_events')
prof_df.createOrReplaceTempView('aml_account_profiles')

print(f'Loaded aml_transaction_events: {tx_df.count():,} rows across {len(tx_df.columns)} columns.')
print(f'Loaded aml_account_profiles:   {prof_df.count():,} rows across {len(prof_df.columns)} columns.')


Wide Column Storage Root: C:\Projects\Mule-tiplier Extended\outputs\wide_column_store


Loaded aml_transaction_events: 6,580 rows across 18 columns.
Loaded aml_account_profiles:   6,751 rows across 20 columns.


## 3. Query Pattern 1: Single Account Wide-Row Lookup ($O(1)$ Hash Partition)

In Cassandra, querying by the partition key `account` directs the coordinator node to the exact replica nodes holding that token in $O(1)$ time without scanning any other partition.


In [3]:
sample_account = prof_df.select('account').first()[0]
print(f"Executing Point Lookup for Partition Key: account = '{sample_account}'")

t0 = time.perf_counter()
single_acct_df = spark.sql(f"""
    SELECT
        account,
        in_degree,
        out_degree,
        ROUND(total_in, 2) AS total_in_usd,
        ROUND(total_out, 2) AS total_out_usd,
        is_sanctioned,
        jurisdiction_code,
        jurisdiction_risk_score,
        is_laundering
    FROM aml_account_profiles
    WHERE account = '{sample_account}'
""")
single_acct_df.show(truncate=False)
print(f"Lookup executed in {(time.perf_counter() - t0)*1000:.2f} ms")


Executing Point Lookup for Partition Key: account = '100428660'


+---------+---------+----------+------------+--------------+-------------+-----------------+-----------------------+-------------+
|account  |in_degree|out_degree|total_in_usd|total_out_usd |is_sanctioned|jurisdiction_code|jurisdiction_risk_score|is_laundering|
+---------+---------+----------+------------+--------------+-------------+-----------------+-----------------------+-------------+
|100428660|0        |444       |0.0         |6.7257973089E8|0            |SE               |0.14                   |1            |
+---------+---------+----------+------------+--------------+-------------+-----------------+-----------------------+-------------+

Lookup executed in 552.93 ms


## 4. Query Pattern 2: Ego-Subgraph Temporal Slicing (Clustering Keys)

Within a physical partition, rows are laid out contiguously by `timestamp DESC`. This allows compliance investigators to inspect the latest 10 transactions for any target account via an indexed forward range slice.


In [4]:
t0 = time.perf_counter()
ego_slice_df = spark.sql(f"""
    SELECT
        timestamp,
        from_account,
        to_account,
        amount_paid_usd,
        is_laundering
    FROM aml_transaction_events
    WHERE from_account = '{sample_account}' OR to_account = '{sample_account}'
    ORDER BY timestamp DESC
    LIMIT 10
""")
ego_slice_df.show(truncate=False)
print(f"Ego-subgraph temporal slice executed in {(time.perf_counter() - t0)*1000:.2f} ms")


+-------------------+------------+----------+---------------+-------------+
|timestamp          |from_account|to_account|amount_paid_usd|is_laundering|
+-------------------+------------+----------+---------------+-------------+
|2022-09-01 00:29:00|100428660   |8002589A0 |26136.82       |0            |
|2022-09-01 00:29:00|100428660   |8002589A0 |26348.38       |0            |
|2022-09-01 00:29:00|100428660   |80025B860 |24662.06       |0            |
|2022-09-01 00:29:00|100428660   |8002D6970 |10.01          |0            |
|2022-09-01 00:29:00|100428660   |800401A80 |8332.37        |0            |
|2022-09-01 00:29:00|100428660   |8008A5CA0 |43393.88       |0            |
|2022-09-01 00:29:00|100428660   |8012835B0 |204.06         |0            |
|2022-09-01 00:29:00|100428660   |80175C9F0 |22588.32       |0            |
|2022-09-01 00:29:00|100428660   |801B5DA80 |11743.88       |0            |
|2022-09-01 00:29:00|100428660   |801B68140 |7791.72        |0            |
+-----------

## 5. Query Pattern 3: High-Risk Cluster Triage (Materialized View)

In Cassandra, Materialized Views replicate data under alternative primary keys. Here we emulate the compliance analyst triage view: accounts with high bidirectional connectivity and significant capital velocity.


In [5]:
t0 = time.perf_counter()
triage_df = spark.sql("""
    SELECT
        account,
        in_degree,
        out_degree,
        ROUND(total_in, 2) AS total_in_usd,
        ROUND(total_out, 2) AS total_out_usd,
        is_sanctioned,
        jurisdiction_risk_score,
        is_laundering
    FROM aml_account_profiles
    WHERE in_degree >= 1 AND out_degree >= 1
    ORDER BY (total_in + total_out) DESC
    LIMIT 10
""")
triage_df.show(truncate=False)
print(f"Triage query executed in {(time.perf_counter() - t0)*1000:.2f} ms")


+---------+---------+----------+--------------+--------------+-------------+-----------------------+-------------+
|account  |in_degree|out_degree|total_in_usd  |total_out_usd |is_sanctioned|jurisdiction_risk_score|is_laundering|
+---------+---------+----------+--------------+--------------+-------------+-----------------------+-------------+
|800CD6CA0|1        |1         |1.329257627E9 |95.25         |0            |0.17                   |0            |
|80064D0F0|1        |1         |8.29552012E8  |102.97        |0            |0.69                   |0            |
|802D2A4D0|1        |1         |15695.45      |7.53318208E8  |0            |0.2                    |0            |
|802D2C030|1        |1         |7.53318208E8  |2.96          |0            |0.11                   |0            |
|8012E17A0|3        |4         |192221.09     |7.4500766307E8|0            |0.99                   |0            |
|80014F510|1        |2         |5.6764725074E8|572.74        |0            |0.28

## 6. Zero-Downtime Schema Evolution

A key requirement in real-world ML engineering is appending newly trained model scores (e.g. `spark_gbt_risk_score`) to existing entity profiles without locking the database or rebuilding multi-terabyte tables.

Below, we simulate Cassandra CQL:
```sql
ALTER TABLE aml.account_profiles ADD spark_gbt_risk_score DOUBLE;
```


In [6]:
# Simulate real-time inference risk score injection
evolved_profiles = prof_df.withColumn(
    'spark_gbt_risk_score',
    F.round(F.rand(seed=42) * 0.95, 4)
)
evolved_profiles.createOrReplaceTempView('aml_account_profiles_v2')

print('Schema evolved dynamically. Top high-risk accounts identified by new model score:')
spark.sql("""
    SELECT
        account,
        total_in,
        jurisdiction_code,
        spark_gbt_risk_score
    FROM aml_account_profiles_v2
    WHERE spark_gbt_risk_score > 0.85
    ORDER BY spark_gbt_risk_score DESC
    LIMIT 10
""").show(truncate=False)


Schema evolved dynamically. Top high-risk accounts identified by new model score:


+---------+---------+-----------------+--------------------+
|account  |total_in |jurisdiction_code|spark_gbt_risk_score|
+---------+---------+-----------------+--------------------+
|80021ADE0|285614.43|IN               |0.95                |
|8001EFA70|0.0      |DE               |0.95                |
|8006489C0|0.0      |RU               |0.9498              |
|8019AC410|0.01     |TR               |0.9495              |
|800055DF0|0.0      |PH               |0.9492              |
|80152E780|0.0      |ES               |0.9491              |
|80D206250|165.68   |CH               |0.9491              |
|8019EA1C0|34569.16 |ID               |0.949               |
|802919450|1028.0   |NL               |0.949               |
|801486850|1128.88  |AF               |0.9487              |
+---------+---------+-----------------+--------------------+



## 7. Architecture Comparison: Wide-Column NoSQL vs Traditional RDBMS

| Dimension | Relational DB (PostgreSQL / MySQL) | Wide-Column NoSQL (Cassandra / Delta Lake) | Why It Matters in AML |
| :--- | :--- | :--- | :--- |
| **Write Throughput** | Limited by B-Tree index updates & single-master WAL | Append-only LSM tree & Parquet bulk ingestion ($O(1)$) | Handles 100k+ global transactions/sec without queue backlog |
| **Data Dimensionality**| Dense schema; sparse columns waste disk / memory | Dynamic wide-column layout; sparse columns store zero bytes | Accommodates 100+ behavioral & graph features per entity |
| **Temporal Traversal** | Recursive multi-table self-joins ($O(N^2)$) | Clustered partition key range scan ($O(\log K)$) | Instantaneous 1-hop and 2-hop ego-network reconstruction |
| **Schema Migrations**  | `ALTER TABLE` acquires exclusive DDL metadata lock | Instant metadata evolution without table rebuilds | Seamless continuous ML feature deployment without downtime |
| **High Availability**  | Failover lag, active-standby replication latency | Masterless peer-to-peer active-active replication | Zero single point of failure across global banking zones |


In [7]:
spark.stop()
print('Spark Database Engine shut down cleanly. Showcase Complete!')


Spark Database Engine shut down cleanly. Showcase Complete!
